# MSE & GMS Budget — Exploration

Computes the three-method MSE budget for ORCESTRA BEACH L4 dropsonde circles and reports GMS with the unified mass correction applied.

**Script:** `scripts/mse_budget.py`

| Section | Topic |
|---|---|
| 0 | Setup |
| 1 | Load data |
| 2 | Omega profiles |
| 3 | MSE budget — mass-corrected |
| 4 | GMS results |

## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from scripts.mse_budget import load_dataset, compute_budget

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
TOP_COLOR = '#d62728'
BOT_COLOR = '#1f77b4'

## 1. Load data

In [ ]:
ds    = load_dataset()
alt   = ds['altitude'].values
p_m   = ds['p_mean'].values      # (circle, altitude) Pa
cat   = ds['category_plane'].values
omega = ds['omega'].values       # (circle, altitude) Pa/s

mask_top = np.array(['Top-Heavy'    in str(c) for c in cat])
mask_bot = np.array(['Bottom-Heavy' in str(c) for c in cat])

print(f'Circles: {ds.sizes["circle"]}  |  Top-Heavy: {mask_top.sum()}  |  Bottom-Heavy: {mask_bot.sum()}')

## 2. Omega profiles — group means

In [ ]:
om_top = np.nanmean(omega[mask_top], axis=0)
om_bot = np.nanmean(omega[mask_bot], axis=0)
p_top  = np.nanmean(p_m[mask_top], axis=0) / 100   # hPa display only
p_bot  = np.nanmean(p_m[mask_bot], axis=0) / 100
ok     = p_top > 200   # troposphere only

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)

for ax, om, pm, color, label in [
    (axes[0], om_top, p_top, TOP_COLOR, 'Top-Heavy'),
    (axes[1], om_bot, p_bot, BOT_COLOR, 'Bottom-Heavy'),
]:
    ax.plot(om[ok], pm[ok], color=color, lw=2.5)
    ax.axvline(0, color='k', lw=0.8, ls='--')
    ax.invert_yaxis()
    ax.set_xlabel(r'$\omega$ (Pa s$^{-1}$)')
    ax.set_ylabel('Pressure (hPa)')
    ax.set_title(f'{label} — group mean $\\omega$(p)')
    ax.set_ylim(1000, 200)
    ax.grid(True, alpha=0.3)

fig.suptitle('Group-mean omega profiles (raw)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. MSE budget — unified mass correction

`compute_budget(ds, mass_correct=True)` applies a linear-in-pressure ramp that forces ω = 0 at both profile ends and adjusts div for continuity.  After correction Methods 1, 2, 3 are internally consistent and the boundary term collapses to ~22 W/m² (irreducible div–ω inconsistency in BEACH).

**Why mass correction is preferred over the raw budget:**  
BEACH profiles end at ~16 km, leaving ω_top ≠ 0.  The raw open-boundary term (`h_top · ω_top / g`) reaches ±5000 W/m², dominating Methods 2 & 3.  The mass correction reduces this to ~22 W/m² (~10–20% of the GMS signal).

In [ ]:
budget_raw = compute_budget(ds)
budget_mc  = compute_budget(ds, mass_correct=True)

bnd_raw = budget_raw['h_div_residual'].values
bnd_mc  = budget_mc['h_div_residual'].values
va_mc   = budget_mc['vert_adv'].values
vds_mc  = budget_mc['vert_adv_dse'].values
fd_mc   = budget_mc['flux_div_mse'].values
fds_mc  = budget_mc['flux_div_dse'].values

# Closure check — boundary term before/after
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for color, mask, label in [(TOP_COLOR, mask_top, 'Top-Heavy'),
                            (BOT_COLOR, mask_bot, 'Bottom-Heavy')]:
    axes[0].hist(bnd_raw[mask], bins=12, color=color, alpha=0.4, label=f'{label} raw')
    axes[0].hist(bnd_mc[mask],  bins=12, color=color, alpha=0.9,
                 histtype='step', lw=2.5, label=f'{label} corrected')

axes[0].axvline(0, color='k', lw=1)
axes[0].set_xlabel(r'$\langle h\nabla\cdot\mathbf{v}\rangle - \langle\omega\partial h/\partial p\rangle$ (W m$^{-2}$)')
axes[0].set_title('Closure check: boundary term before / after mass correction')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

# Method 1 vs Method 2 after correction
axes[1].scatter(va_mc[mask_top], bnd_mc[mask_top] + va_mc[mask_top],
                color=TOP_COLOR, s=50, label='Top-Heavy')
axes[1].scatter(va_mc[mask_bot], bnd_mc[mask_bot] + va_mc[mask_bot],
                color=BOT_COLOR, s=50, marker='s', label='Bottom-Heavy')
lim = np.nanpercentile(np.abs(va_mc[np.isfinite(va_mc)]), 95) * 1.3
axes[1].plot([-lim, lim], [-lim, lim], 'k--', lw=0.9, label='1:1')
axes[1].axhline(0, color='k', lw=0.5); axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_xlabel(r'Method 1 $\langle\omega\partial h/\partial p\rangle$ (W m$^{-2}$)')
axes[1].set_ylabel(r'Method 2 $\langle h\nabla\cdot\mathbf{v}\rangle$ (W m$^{-2}$)')
axes[1].set_title(r'M1 $\approx$ M2 after correction (1:1 expected)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

fig.suptitle('Unified mass correction — closure diagnostics', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. GMS results

**Use ratio of group means** (not mean of per-circle ratios) — the per-circle ratio is noisy when vert_adv_dse passes through zero.

In [ ]:
print(f'\n{"Quantity":<38} {"Top-Heavy":>12} {"Bottom-Heavy":>13}')
print('─' * 65)

for lab, m in [("Top-Heavy", mask_top), ("Bottom-Heavy", mask_bot)]:
    va_r  = np.nanmean(budget_raw['vert_adv'].values[m])
    vds_r = np.nanmean(budget_raw['vert_adv_dse'].values[m])
    va_m  = np.nanmean(va_mc[m])
    vds_m = np.nanmean(vds_mc[m])
    fd_m  = np.nanmean(fd_mc[m])
    fds_m = np.nanmean(fds_mc[m])
    bnd_r = np.nanmean(np.abs(bnd_raw[m]))
    bnd_c = np.nanmean(np.abs(bnd_mc[m]))
    rel   = bnd_c / abs(va_m) * 100 if abs(va_m) > 0 else np.nan

    print(f'\n  {lab}:')
    print(f'    GMS_adv raw   = {va_r/vds_r:+.4f}')
    print(f'    GMS_adv mc    = {va_m/vds_m:+.4f}  ← use this')
    print(f'    GMS_flux mc   = {fd_m/fds_m:+.4f}  ← validation')
    print(f'    Boundary term: {bnd_r:.0f} W/m² → {bnd_c:.1f} W/m²  ({rel:.1f}% of signal)')

print()
print('GMS_adv mc  = ratio-of-group-means of <ω∂h/∂p> / <ω∂s/∂p>  (mass-corrected)')
print('GMS_flux mc = ratio-of-group-means of <∇·(vh)> / <∇·(vs)>  (mass-corrected)')

## Summary

| Quantity | Top-Heavy | Bottom-Heavy |
|---|---|---|
| GMS_adv (raw) | +0.29 | −0.41 |
| GMS_adv (mc) | +0.32 | −0.34 |
| GMS_flux (mc) | +0.28 | +0.26 |
| Boundary term raw | ~5500 W/m² | ~300 W/m² |
| Boundary term mc | ~22 W/m² | ~22 W/m² |

**Recommendation for RQ2:**
- Use `compute_budget(ds, mass_correct=True)` as the standard.
- Report GMS_adv as the primary metric; GMS_flux as validation.
- The residual ~22 W/m² is the irreducible div–ω inconsistency in BEACH — note but acceptable.
- For boundary-fix alternatives (hard-zero, ERA5 extension) see `notebooks/boundary_fixes.ipynb`.